# Part 3. Feasible Growth (v8, 2026-06-01)
## Smooth ordering-specific reconstruction and re-screening

Analytical tables are built by `code/build_part3_*.py`; this notebook
loads those v8 outputs (Cell A) and renders Figures P3-1/P3-2/P3-3.

## Cell A. Setup and loading

Load Part-1 deterministic outputs, Part-2 shortfall windows and screened
paths under both orderings, and build the failed ordering-specific
universe (78 rows = 39 failed cases x 2 orderings).

In [ ]:
from __future__ import annotations
import math
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# v8 constants
T_BASE, T_END = 2030, 2179
YEARS = np.arange(T_BASE, T_END + 1)
PHI   = (3.0 + math.sqrt(5.0)) / 2.0
SQRT3 = math.sqrt(3.0)
EXCEED_EPS_GT = 1e-4
KEY = ['country', 'scenario', 'model', 'ordering']

ROOT     = Path('/Users/xg320/Desktop/IC-Sam-works/paper/Iman-2026')
P3_DIR   = ROOT / '03_feasible_growth'
OUT_DIR  = P3_DIR / 'output'
FIG_DIR  = OUT_DIR / 'figures'
CTRY_DIR = FIG_DIR / 'country_profiles'
for d in (FIG_DIR, CTRY_DIR):
    d.mkdir(parents=True, exist_ok=True)

# Load the v8 analytical outputs produced by code/build_part3_*.py
# (Step 1: build_part3_input_table.py; reconstruction: build_part3_reconstruction.py;
#  rescreen: build_part3_rescreen.py.) The figure cells below consume `failed`
# (one row per failed ordering-specific case) and `envelopes_gt`.
failed = pd.read_csv(OUT_DIR / 'smooth_reconstruction_summary.csv').rename(
    columns={'S0_gt': 'S_0', 'C_scenario_gt': 'C_scenario'})

_ts = pd.read_csv(OUT_DIR / 'smooth_reconstruction_timeseries.csv')
envelopes_gt = {}
for k, g in _ts[_ts.curve_kind == 'screened_envelope'].groupby(KEY):
    envelopes_gt[tuple(k)] = g.sort_values('year')['rate_mt_yr'].to_numpy() / 1000.0

# optional: re-screen results (for any rescreen-aware annotations)
try:
    rescreen = pd.read_csv(OUT_DIR / 'reconstructed_rescreen_summary.csv')
except FileNotFoundError:
    rescreen = None

solved = failed[~failed['landmark_infeasible']]
print(f'Loaded v8 Part-3 outputs: {len(failed)} failed cases '
      f'({len(solved)} reconstructed), {len(envelopes_gt)} envelopes.')
print(f'  failed countries: {sorted(failed.country.unique())}')
print(f'  by ordering: {failed.ordering.value_counts().to_dict()}')


## Cell B. Curve helpers

Scalar Logistic / Gompertz forward and inverse, using the same equations and
CAGR definition as the active Part-1 V8 growth model. All math in Gt/yr.

In [ ]:
# Inverse (CAGR g to ODE r)
def logistic_inv(C, S0, t0, g):
    """Returns (r, tn, tp) for Logistic given CAGR g."""
    k    = (C - S0) / S0
    S_Tn = C / (3.0 + SQRT3)
    A    = math.log(k) - math.log(2.0 + SQRT3)
    r    = A * math.log(1.0 + g) / math.log(S_Tn / S0)
    return r, t0 + A / r, t0 + math.log(k) / r


def gompertz_inv(C, S0, t0, g):
    """Returns (r, b, tn, tp) for Gompertz given CAGR g."""
    b    = -math.log(S0 / C)
    P_Tn = C * math.exp(-PHI)
    B    = math.log(b / PHI)
    r    = B * math.log(1.0 + g) / math.log(P_Tn / S0)
    return r, b, t0 + B / r, t0 + math.log(b) / r


# Forward (r-parameterised) evaluators
def logistic_rate_at(year_or_arr, C, r, S0, t0=T_BASE):
    k = (C - S0) / S0
    S = C / (1.0 + k * np.exp(-r * (year_or_arr - t0)))
    return r * S * (1.0 - S / C)


def gompertz_rate_at(year_or_arr, C, b, r, t0=T_BASE):
    tau = year_or_arr - t0
    s   = b * np.exp(-r * tau)
    return C * r * s * np.exp(-s)


def logistic_cum_at(year, C, r, S0, t0=T_BASE):
    """Analytical Logistic cumulative storage at year."""
    k = (C - S0) / S0
    return C / (1.0 + k * np.exp(-r * (year - t0)))


def gompertz_cum_at(year, C, b, r, t0=T_BASE):
    """Analytical Gompertz cumulative storage at year."""
    tau = year - t0
    return C * np.exp(-b * np.exp(-r * tau))


def logistic_peak_year(C, r, S0, t0=T_BASE):
    k = (C - S0) / S0
    return t0 + math.log(k) / r


def gompertz_peak_year(C, S0, r, t0=T_BASE):
    b = -math.log(S0 / C)
    return t0 + math.log(b) / r


# Forward in g-space (uses inverse then evaluates)
def forward_g(model, C, S0, g, years_arr=YEARS):
    if model == 'Logistic':
        r, tn, tp = logistic_inv(C, S0, T_BASE, g)
        rate = logistic_rate_at(years_arr, C, r, S0)
    else:
        r, b, tn, tp = gompertz_inv(C, S0, T_BASE, g)
        rate = gompertz_rate_at(years_arr, C, b, r)
    cum = S0 + np.concatenate([[0], np.cumsum(rate[:-1])])  # numerical
    return {'r': r, 'tn': tn, 'tp': tp, 'rate': rate, 'cum': cum}


# Forward in r-space (used by the reconstruction solver)
def forward_r(model, C, S0, r, years_arr=YEARS):
    if model == 'Logistic':
        rate = logistic_rate_at(years_arr, C, r, S0)
        tp   = logistic_peak_year(C, r, S0)
    else:
        b = -math.log(S0 / C)
        rate = gompertz_rate_at(years_arr, C, b, r)
        tp   = gompertz_peak_year(C, S0, r)
    cum = S0 + np.concatenate([[0], np.cumsum(rate[:-1])])
    return {'rate': rate, 'cum': cum, 'tp': tp}


# r to g helpers
def r_to_g_logistic(C, S0, r):
    k    = (C - S0) / S0
    S_Tn = C / (3.0 + SQRT3)
    A    = math.log(k) - math.log(2.0 + SQRT3)
    if A <= 0 or math.log(S_Tn / S0) <= 0:
        return float('nan')
    return math.exp(r * math.log(S_Tn / S0) / A) - 1.0


def r_to_g_gompertz(C, S0, r):
    b    = -math.log(S0 / C)
    P_Tn = C * math.exp(-PHI)
    B    = math.log(b / PHI)
    if B <= 0 or math.log(P_Tn / S0) <= 0:
        return float('nan')
    return math.exp(r * math.log(P_Tn / S0) / B) - 1.0


def r_to_g(model, C, S0, r):
    if model == 'Logistic':
        return r_to_g_logistic(C, S0, r)
    return r_to_g_gompertz(C, S0, r)


print('Curve helpers ready.')

# Spot-check: reproduce Australia/reference/Logistic original
_row = failed[(failed['country']=='Australia') & (failed['scenario']=='reference')
              & (failed['model']=='Logistic')].iloc[0]
_chk = forward_g(_row['model'], _row['C_scenario'], _row['S_0'], _row['g_original'])
print(f'\nSpot-check Australia · reference · Logistic:')
print(f'  r_computed  = {_chk["r"]:.6f}  vs Part-1 r_L     = {_row["r_original"]:.6f}')
print(f'  tp_computed = {_chk["tp"]:.2f}  vs Part-1 tp_L    = {_row["tp_original"]:.2f}')
print(f'  rate@2050   = {logistic_rate_at(2050, _row["C_scenario"], _chk["r"], _row["S_0"]):.6f} '
      f'vs Part-1 rate_2050 = {_row["rate_2050_original"]:.6f}')

## Cell C. Extract ordering-specific screened envelope and peak landmark

For each failed `(country, scenario, model, ordering)` case, pull
`screened_rate(y)` from Part 2 over 2030 to 2179 and compute the peak
landmark `R_peak_order = max_y screened_rate(y)`. All values converted
to Gt/yr (Part 2 stores Mt/yr).

In [3]:
# (v8) The smooth reconstruction, envelope diagnostics, CSV outputs, and
# re-screening are now produced by the reproducible scripts under
# 03_feasible_growth/code/ and loaded in Cell A. This cell is intentionally
# a no-op so the figure cells below run directly on the loaded v8 outputs.
pass


## Cell D. Solve smooth 3-landmark reconstruction

For each failed ordering-specific case, solve the 2-equation system in
`(C, r)`:

```
peak_rate(model, C, r) = R_peak_order
rate_at_2050(model, C, S0, r) = R2050
```

Substituting the peak constraint reduces this to a 1-variable equation
in `r`:

- Logistic: `C(r) = 4·R_peak / r`
- Gompertz: `C(r) = e·R_peak / r`

The resulting `f(r) = rate_at_2050(C(r), S0, r) − R2050` is non-monotonic
(rises, peaks, falls), so it admits two roots in general. We scan `r`
over a fine grid, find sign changes, brentq each bracket, then apply
§4.4's branch-selection rule: drop roots with `peak_year < 2050`, and
among the rest keep the one with smallest envelope inconsistency.

In [4]:
# (v8) The smooth reconstruction, envelope diagnostics, CSV outputs, and
# re-screening are now produced by the reproducible scripts under
# 03_feasible_growth/code/ and loaded in Cell A. This cell is intentionally
# a no-op so the figure cells below run directly on the loaded v8 outputs.
pass


## Cell E. Envelope consistency diagnostics

The 3-landmark reconstruction only matches the curve at 2030/2050/peak.
The rest of the curve is determined by the model family, so the
reconstructed curve may exceed the full year-by-year envelope at
intermediate years.

For each reconstructed case compute the agreement between
`rate_recon_order(y)` and `E_order(y)`:

- `max_envelope_exceedance_mt_yr`
- `years_exceeding_envelope`
- `first_exceed_year`, `last_exceed_year`
- `rmse_to_envelope_mt_yr`
- `envelope_consistent` = `years_exceeding_envelope == 0`

In [5]:
# (v8) The smooth reconstruction, envelope diagnostics, CSV outputs, and
# re-screening are now produced by the reproducible scripts under
# 03_feasible_growth/code/ and loaded in Cell A. This cell is intentionally
# a no-op so the figure cells below run directly on the loaded v8 outputs.
pass


## Cell F. Write CSV outputs

Two CSVs in `output/`:

- `smooth_reconstruction_summary.csv`, one row per failed
  ordering-specific case, all original + landmark + reconstructed +
  consistency fields per §9.1 of the notes.
- `smooth_reconstruction_timeseries.csv`, long format,
  `curve_kind ∈ {original, screened_envelope, reconstructed}`,
  35,100 rows (78 cases x 3 curve kinds x 150 years).

In [6]:
# (v8) The smooth reconstruction, envelope diagnostics, CSV outputs, and
# re-screening are now produced by the reproducible scripts under
# 03_feasible_growth/code/ and loaded in Cell A. This cell is intentionally
# a no-op so the figure cells below run directly on the loaded v8 outputs.
pass


## Figure P3-1. Envelope-fit honesty diagnostic (supplementary)

Single-panel scatter answering the reviewer question: the
reconstruction is anchored to three landmarks, so how far from the
full envelope does it drift in the years between them?

- x = `years_exceeding_envelope` (count over 2030 to 2179)
- y = `max_envelope_exceedance_mt_yr`
- colour = ordering, marker = model

The `C_scenario` vs `C_recon` panel is not shown here; that information lives in Fig P3-3 panel (a) in a much more readable per-country form.

In [7]:
plt.rcParams.update({
    'font.size': 11, 'axes.titlesize': 12, 'axes.labelsize': 11,
    'xtick.labelsize': 9.5, 'ytick.labelsize': 9.5, 'legend.fontsize': 9,
    'axes.spines.top': False, 'axes.spines.right': False,
})
ORDER_COLOR  = {'ascending': '#4e79a7', 'descending': '#e15759'}
MODEL_MARKER = {'Logistic': 'o', 'Gompertz': '^'}

main = failed[~failed['landmark_infeasible']].copy()

fig, ax = plt.subplots(figsize=(8.5, 6.5))

for ord_ in ('ascending', 'descending'):
    for mdl in ('Logistic', 'Gompertz'):
        sub = main[(main['ordering'] == ord_) & (main['model'] == mdl)]
        if sub.empty:
            continue
        ax.scatter(sub['years_exceeding_envelope'],
                    sub['max_envelope_exceedance_mt_yr'],
                    s=85, marker=MODEL_MARKER[mdl],
                    facecolors=ORDER_COLOR[ord_], edgecolors='white',
                    linewidth=0.7, alpha=0.85, zorder=5)

ax.set_xlabel('years_exceeding_envelope  (count over 2030–2179)')
ax.set_ylabel('max_envelope_exceedance  (Mt CO₂/yr)')
ax.grid(True, linestyle=':', lw=0.5, alpha=0.4)
fig.suptitle('Figure P3-1 (supplementary) — '
              'Envelope-fit honesty diagnostic',
              fontsize=13, fontweight='bold', y=0.975)

# Italic subtitle stats
n         = len(main)
n_consist = int(main['envelope_consistent'].sum())
med_exc_y = main['years_exceeding_envelope'].median()
med_rmse  = main['rmse_to_envelope_mt_yr'].median()
fig.text(0.5, 0.925,
    f'Across {n} reconstructed cases: '
    f'envelope-consistent = {n_consist}/{n} · '
    f'median years exceeding = {med_exc_y:.0f} · '
    f'median RMSE = {med_rmse:.1f} Mt/yr.',
    ha='center', va='top', fontsize=10, style='italic', color='#444')

# Legend
ord_h = [plt.Line2D([0], [0], marker='s', linestyle='', markersize=10,
                      markerfacecolor=ORDER_COLOR[o],
                      markeredgecolor='white', label=o)
          for o in ('ascending', 'descending')]
mdl_h = [plt.Line2D([0], [0], marker=MODEL_MARKER[m], linestyle='',
                      markersize=8, markerfacecolor='#888',
                      markeredgecolor='white', label=m)
          for m in ('Logistic', 'Gompertz')]
fig.legend(handles=ord_h + mdl_h,
            loc='lower center', bbox_to_anchor=(0.5, -0.02),
            ncol=4, frameon=False, fontsize=10)

plt.subplots_adjust(top=0.86, bottom=0.16)
fig.savefig(FIG_DIR / 'fig_envelope_consistency.pdf',
             dpi=180, bbox_inches='tight')
fig.savefig(FIG_DIR / 'fig_envelope_consistency.png',
             dpi=180, bbox_inches='tight')
plt.close(fig)
print(f'wrote {FIG_DIR / "fig_envelope_consistency"}.{{pdf,png}}')

wrote /Users/xg320/Desktop/IC-Sam-works/paper/Iman-2026/03_feasible_growth/output/figures/fig_envelope_consistency.{pdf,png}


## Figure P3-3. Country-level feasibility summary (paper main)

Two-panel composite for the paper main text.

Panel (a) is a feasibility cost forest plot per country×model. Rows are
country×model groups (only those with failed cases); each row shows all
failed-scenario `C_recon/C_scenario` ratios as a strip plus median
marker plus min/max whiskers, coloured by ordering, with a reference
line at 1.0.

Panel (b) is original vs feasible curve sparklines, one per failed
country. Representative scenario rule: `reference` if it failed, else
the failed scenario with the smallest `C_recon/C_scenario` ratio.
Curves are original (grey dashed), feasible_asc (blue solid),
feasible_desc (red solid).

In [ ]:
# Figure P3-3: country-level feasibility summary (paper main figure)
# Layout:
#   Row 0: forest plot of C_recon/C_scenario per (country, model)
#   Row 1: Logistic sparklines per country  (model-specific representative)
#   Row 2: Gompertz sparklines per country  (model-specific representative)

ORDER_COLOR  = {'ascending': '#4e79a7', 'descending': '#e15759'}
MODEL_MARKER = {'Logistic': 'o', 'Gompertz': '^'}

main = failed[~failed['landmark_infeasible']].copy()


# Representative scenario per (country, model)
def _rep_scenario(country, mdl):
    sub = main[(main['country'] == country) & (main['model'] == mdl)]
    if sub.empty:
        return None
    if 'reference' in set(sub['scenario']):
        return 'reference'
    return sub.loc[sub['C_ratio_recon_to_scenario'].idxmin(), 'scenario']


countries_in_failed = sorted(main['country'].unique())
print(f'Countries with failed cases: {countries_in_failed}')
rep_map = {(c, m): _rep_scenario(c, m)
            for c in countries_in_failed
            for m in ('Logistic', 'Gompertz')}
for k, v in rep_map.items():
    print(f'  {k[0]:>10s} · {k[1]:<9s} → {v}')


# Layout: 3 rows x n_countries cols
n_countries = len(countries_in_failed)
fig = plt.figure(figsize=(15, 13.5))
gs = fig.add_gridspec(
    3, n_countries,
    height_ratios=[2.2, 1.0, 1.0],
    left=0.07, right=0.985, top=0.92, bottom=0.085,
    hspace=0.55, wspace=0.18)

# Panel (a): forest plot (spans full width)
ax_a = fig.add_subplot(gs[0, :])

# Build country x model rows (Logistic top, Gompertz bottom within country)
row_labels = []
for c in countries_in_failed:
    for m in ('Logistic', 'Gompertz'):
        sub = main[(main['country'] == c) & (main['model'] == m)]
        if not sub.empty:
            row_labels.append((c, m))
y_positions = np.arange(len(row_labels))[::-1]

for y_pos, (c, m) in zip(y_positions, row_labels):
    sub = main[(main['country'] == c) & (main['model'] == m)]
    vals = sub['C_ratio_recon_to_scenario'].values
    ax_a.hlines(y_pos, vals.min(), vals.max(),
                 colors='#888', lw=1.0, alpha=0.6, zorder=1)
    for _, r in sub.iterrows():
        ax_a.scatter(r['C_ratio_recon_to_scenario'], y_pos,
                      s=55,
                      marker=MODEL_MARKER[m],
                      facecolors=ORDER_COLOR[r['ordering']],
                      edgecolors='white', linewidth=0.6,
                      alpha=0.75, zorder=3)
    median = float(np.median(vals))
    ax_a.scatter(median, y_pos, s=140, marker='|',
                  color='#222', linewidth=2.4, zorder=5)

ax_a.axvline(1.0, color='#444', linestyle='--', lw=1.0, alpha=0.7, zorder=0)
ax_a.text(0.995, 0.965,
           'C_recon = C_scenario\n(no reduction)',
           transform=ax_a.get_xaxis_transform(),
           ha='right', va='top', fontsize=8.5, color='#444', style='italic')

ax_a.set_yticks(y_positions)
ax_a.set_yticklabels([f'{c} · {m}' for (c, m) in row_labels], fontsize=10)
ax_a.set_xlabel('Implied long-term commitment ratio  '
                'C_recon / C_scenario', fontsize=11)
ax_a.set_xlim(0, max(1.05, main['C_ratio_recon_to_scenario'].max() * 1.1))
ax_a.set_ylim(-0.5, len(row_labels) - 0.5)
ax_a.set_title('(a) Feasibility cost per country × model — '
                'strip = scenarios, long tick = median, '
                'whiskers = min/max',
                fontsize=12, fontweight='bold', loc='left', pad=10)
ax_a.grid(True, axis='x', linestyle=':', lw=0.5, alpha=0.4)


# Panel (b): sparklines, Logistic row + Gompertz row
def _draw_sparkline(ax, country, mdl):
    scenario = rep_map[(country, mdl)]
    if scenario is None:
        ax.text(0.5, 0.5,
                f'{mdl} passed Part 2\nin every scenario',
                transform=ax.transAxes, ha='center', va='center',
                fontsize=8.5, color='#888', style='italic')
        ax.set_xticks([]); ax.set_yticks([])
        for s_ in ax.spines.values():
            s_.set_visible(False)
        ax.set_title(f'{country}',
                      fontsize=9.5, fontweight='bold', pad=4)
        return

    sub_cs = failed[(failed['country'] == country) &
                     (failed['scenario'] == scenario) &
                     (failed['model'] == mdl)]
    if sub_cs.empty:
        ax.set_visible(False); return

    # Original
    row_any = sub_cs.iloc[0]
    fwd_orig = forward_g(mdl, row_any['C_scenario'], row_any['S_0'],
                          row_any['g_original'])
    ax.plot(YEARS, fwd_orig['rate'] * 1000.0,
             color='#444', linestyle='--', lw=1.0, alpha=0.85, zorder=2)

    # Feasible curves under each ordering
    for ord_, color in (('ascending', '#4e79a7'),
                         ('descending', '#e15759')):
        sub_o = sub_cs[sub_cs['ordering'] == ord_]
        if sub_o.empty or sub_o.iloc[0]['landmark_infeasible']:
            continue
        ro = sub_o.iloc[0]
        fwd = forward_r(mdl, ro['C_recon_order_gt'], ro['S_0'],
                         ro['r_recon_order'])
        ax.plot(YEARS, fwd['rate'] * 1000.0, color=color, lw=1.6,
                 alpha=0.9, zorder=3)

    ax.set_xlim(2030, 2179)
    ax.set_ylim(bottom=0)
    ax.set_xticks([2030, 2100, 2179])
    ax.set_xticklabels(['2030', '2100', '2179'], fontsize=8)
    ax.tick_params(axis='y', labelsize=8)
    ax.set_title(f'{country}  ({scenario})',
                  fontsize=9.5, fontweight='bold', pad=4)
    ax.grid(True, linestyle=':', lw=0.3, alpha=0.4)
    for sp in ('top', 'right'):
        ax.spines[sp].set_visible(False)


# Logistic row
for col_idx, country in enumerate(countries_in_failed):
    ax = fig.add_subplot(gs[1, col_idx])
    _draw_sparkline(ax, country, 'Logistic')
    if col_idx == 0:
        ax.set_ylabel('Logistic\nrate (Mt CO₂/yr)', fontsize=9.5,
                       fontweight='bold')

# Gompertz row
for col_idx, country in enumerate(countries_in_failed):
    ax = fig.add_subplot(gs[2, col_idx])
    _draw_sparkline(ax, country, 'Gompertz')
    if col_idx == 0:
        ax.set_ylabel('Gompertz\nrate (Mt CO₂/yr)', fontsize=9.5,
                       fontweight='bold')


# Italic subtitle stats
med_ratio = main['C_ratio_recon_to_scenario'].median()
fig.text(0.5, 0.948,
    f'Smooth-reconstructed long-term commitment '
    f'C_recon/C_scenario across {len(main)} failed order-cases · '
    f'median = {med_ratio:.2f} · sparklines show one representative '
    f'scenario per (country, model): reference if failed, else '
    f'worst-clipped.',
    ha='center', va='top', fontsize=9.5, style='italic', color='#444')


# Legend
ord_h = [plt.Line2D([0], [0], marker='s', linestyle='', markersize=10,
                      markerfacecolor=ORDER_COLOR[o],
                      markeredgecolor='white', label=o)
          for o in ('ascending', 'descending')]
mdl_h = [plt.Line2D([0], [0], marker=MODEL_MARKER[m], linestyle='',
                      markersize=8, markerfacecolor='#888',
                      markeredgecolor='white', label=m)
          for m in ('Logistic', 'Gompertz')]
curve_h = [
    plt.Line2D([0], [0], linestyle='--', color='#444', lw=1.4,
                label='original (Part-1)'),
    plt.Line2D([0], [0], linestyle='-', color='#4e79a7', lw=1.8,
                label='feasible · ascending'),
    plt.Line2D([0], [0], linestyle='-', color='#e15759', lw=1.8,
                label='feasible · descending'),
]
fig.legend(handles=ord_h + mdl_h + curve_h,
            loc='lower center', bbox_to_anchor=(0.5, 0.005),
            ncol=7, frameon=False, fontsize=9)

fig.suptitle('Figure P3-3 — Country-level feasibility summary',
              fontsize=15, fontweight='bold', y=0.985)

fig.savefig(FIG_DIR / 'fig_country_feasibility_summary.pdf',
             dpi=180, bbox_inches='tight')
fig.savefig(FIG_DIR / 'fig_country_feasibility_summary.png',
             dpi=180, bbox_inches='tight')
plt.close(fig)
print(f'wrote {FIG_DIR / "fig_country_feasibility_summary"}.{{pdf,png}}')

## Figure P3-2. Per-(country, scenario) reconstruction

One figure per failed (country, scenario), Logistic and Gompertz
side-by-side. Each panel:

- grey dashed: original Part-1 curve
- blue thin dashed: ascending screened envelope
- red thin dashed: descending screened envelope
- blue solid: ascending smooth reconstruction
- red solid: descending smooth reconstruction
- markers at 2050 and at the reconstructed peak

The basin replay is written to `reconstructed_rescreen_summary.csv` and `reconstructed_assignment.csv`; the profile panels focus on the growth-curve comparison.

In [9]:
def _orig_metrics(row):
    """Compute the 6 comparison metrics for the original Part-1 curve.

    Returns dict in human-readable units: CAGR %, years, Mt/yr, Gt.
    """
    mdl = row['model']; C = row['C_scenario']; S0 = row['S_0']
    g   = row['g_original']
    if mdl == 'Logistic':
        r, tn, tp = logistic_inv(C, S0, T_BASE, g)
        peak_rate = r * C / 4.0
        cum_2179  = logistic_cum_at(2179.0, C, r, S0)
        rate_tn   = logistic_rate_at(tn, C, r, S0)
    else:
        r, b, tn, tp = gompertz_inv(C, S0, T_BASE, g)
        peak_rate = r * C / math.e
        cum_2179  = gompertz_cum_at(2179.0, C, b, r)
        rate_tn   = gompertz_rate_at(tn, C, b, r)
    return {'g_pct': g * 100.0, 'tn': tn, 'rate_tn_mt': rate_tn * 1000.0,
            'tp': tp, 'peak_rate_mt': peak_rate * 1000.0,
            'cum_2179_gt': cum_2179, 'resource_C_gt': C}


def _recon_metrics(row):
    """Compute the 6 comparison metrics for a reconstructed curve."""
    if row.get('landmark_infeasible', False):
        return None
    mdl = row['model']; C = row['C_recon_order_gt']; S0 = row['S_0']
    r   = row['r_recon_order']
    g   = row['g_recon_order']
    tn  = row['tn_recon_order']
    tp  = row['tp_recon_order']
    if mdl == 'Logistic':
        peak_rate = r * C / 4.0
        cum_2179  = logistic_cum_at(2179.0, C, r, S0)
        rate_tn   = logistic_rate_at(tn, C, r, S0)
    else:
        b = -math.log(S0 / C)
        peak_rate = r * C / math.e
        cum_2179  = gompertz_cum_at(2179.0, C, b, r)
        rate_tn   = gompertz_rate_at(tn, C, b, r)
    return {'g_pct': g * 100.0, 'tn': tn, 'rate_tn_mt': rate_tn * 1000.0,
            'tp': tp, 'peak_rate_mt': peak_rate * 1000.0,
            'cum_2179_gt': cum_2179, 'resource_C_gt': C}


def _format_comparison_table(m_orig, m_asc, m_desc):
    """Format the original vs feasible_asc vs feasible_desc comparison.

    Each column uses a fixed width so the monospace layout is aligned.
    """
    def _f(v, fmt):
        if v is None: return f'{"  —":>9}'
        return f'{v:{fmt}}'

    rows = [
        f'{"":<18}{"original":>9}{"asc":>9}{"desc":>9}',
        '─' * 45,
        f'{"CAGR g (%)":<18}'
        f'{_f(m_orig["g_pct"], "9.2f")}'
        f'{_f(m_asc["g_pct"] if m_asc else None, "9.2f")}'
        f'{_f(m_desc["g_pct"] if m_desc else None, "9.2f")}',
        f'{"inflection year":<18}'
        f'{_f(m_orig["tn"], "9.0f")}'
        f'{_f(m_asc["tn"] if m_asc else None, "9.0f")}'
        f'{_f(m_desc["tn"] if m_desc else None, "9.0f")}',
        f'{"infl rate (Mt/yr)":<18}'
        f'{_f(m_orig["rate_tn_mt"], "9.0f")}'
        f'{_f(m_asc["rate_tn_mt"] if m_asc else None, "9.0f")}'
        f'{_f(m_desc["rate_tn_mt"] if m_desc else None, "9.0f")}',
        f'{"peak year":<18}'
        f'{_f(m_orig["tp"], "9.0f")}'
        f'{_f(m_asc["tp"] if m_asc else None, "9.0f")}'
        f'{_f(m_desc["tp"] if m_desc else None, "9.0f")}',
        f'{"peak rate (Mt/yr)":<18}'
        f'{_f(m_orig["peak_rate_mt"], "9.0f")}'
        f'{_f(m_asc["peak_rate_mt"] if m_asc else None, "9.0f")}'
        f'{_f(m_desc["peak_rate_mt"] if m_desc else None, "9.0f")}',
        f'{"resource C (Gt)":<18}'
        f'{_f(m_orig["resource_C_gt"], "9.1f")}'
        f'{_f(m_asc["resource_C_gt"] if m_asc else None, "9.1f")}'
        f'{_f(m_desc["resource_C_gt"] if m_desc else None, "9.1f")}',
    ]
    return '\n'.join(rows)


def _plot_country_scenario(country, scenario):
    sub_cs = failed[(failed['country'] == country) &
                     (failed['scenario'] == scenario)]
    if sub_cs.empty:
        return

    # Nested GridSpec: outer 1×2 for Logistic | Gompertz. Each cell
    # holds a sub-grid with the curve plot on the left and a dedicated
    # table area on the right — no overlap possible.
    fig = plt.figure(figsize=(18, 7.5))
    outer = fig.add_gridspec(1, 2,
                              left=0.05, right=0.985,
                              top=0.90, bottom=0.18,
                              wspace=0.18)

    for ax_idx, mdl in enumerate(('Logistic', 'Gompertz')):
        sub_m = sub_cs[sub_cs['model'] == mdl]

        if sub_m.empty:
            # Placeholder spans both plot + table columns
            ax = fig.add_subplot(outer[0, ax_idx])
            ax.text(0.5, 0.5,
                    f'{mdl}\npassed Part 2 — not in failed universe',
                    transform=ax.transAxes, ha='center', va='center',
                    fontsize=13, color='#666', style='italic')
            ax.set_xticks([]); ax.set_yticks([])
            for s in ax.spines.values():
                s.set_visible(False)
            ax.set_title(mdl, fontsize=12, fontweight='bold', loc='left')
            continue

        # Split this cell into plot (left ~70%) and table (right ~30%)
        inner = outer[0, ax_idx].subgridspec(1, 2,
                                              width_ratios=[2.2, 1.3],
                                              wspace=0.10)
        ax = fig.add_subplot(inner[0, 0])
        ax_tbl = fig.add_subplot(inner[0, 1])
        ax_tbl.axis('off')

        # Original curve
        row_any = sub_m.iloc[0]
        fwd_orig = forward_g(mdl, row_any['C_scenario'], row_any['S_0'],
                              row_any['g_original'])
        ax.plot(YEARS, fwd_orig['rate'] * 1000.0, color='#444',
                 linestyle='--', lw=1.4, alpha=0.85,
                 label='original (Part-1)', zorder=4)
        i50 = int(np.where(YEARS == 2050)[0][0])
        ax.plot(2050, fwd_orig['rate'][i50] * 1000.0,
                 marker='o', markersize=4, markerfacecolor='#444',
                 markeredgecolor='white', zorder=6)

        # Per-ordering envelope + reconstruction
        for ord_, color in (('ascending', '#4e79a7'),
                             ('descending', '#e15759')):
            sub_o = sub_m[sub_m['ordering'] == ord_]
            if sub_o.empty: continue
            row_o = sub_o.iloc[0]
            case_key = tuple(row_o[k] for k in KEY)
            env = envelopes_gt.get(case_key)
            if env is not None:
                ax.plot(YEARS, env * 1000.0, color=color, linestyle=':',
                         lw=1.0, alpha=0.6,
                         label=f'envelope · {ord_}', zorder=2)
            if not row_o['landmark_infeasible']:
                fwd_rec = forward_r(mdl, row_o['C_recon_order_gt'],
                                     row_o['S_0'], row_o['r_recon_order'])
                ax.plot(YEARS, fwd_rec['rate'] * 1000.0, color=color,
                         lw=2.0, alpha=0.95,
                         label=f'reconstructed · {ord_}', zorder=5)
                ax.plot(2050, fwd_rec['rate'][i50] * 1000.0,
                         marker='o', markersize=4, markerfacecolor=color,
                         markeredgecolor='white', zorder=6)
                py = row_o['peak_year_recon_order']
                if pd.notna(py):
                    py_i = int(py - T_BASE)
                    if 0 <= py_i < len(YEARS):
                        ax.plot(py, fwd_rec['rate'][py_i] * 1000.0,
                                 marker='*', markersize=10,
                                 markerfacecolor=color,
                                 markeredgecolor='white', zorder=7)

        # Cosmetic
        ax.set_xlim(2030, 2179)
        ax.set_ylim(bottom=0)
        ax.set_xlabel('Year')
        ax.set_ylabel('Annual rate (Mt CO₂/yr)')
        ax.set_title(mdl, fontsize=12, fontweight='bold', loc='left')
        ax.grid(True, linestyle=':', lw=0.4, alpha=0.4)

        # Build comparison table
        m_orig = _orig_metrics(row_any)
        row_a = sub_m[sub_m['ordering'] == 'ascending']
        row_d = sub_m[sub_m['ordering'] == 'descending']
        m_asc  = _recon_metrics(row_a.iloc[0])  if not row_a.empty else None
        m_desc = _recon_metrics(row_d.iloc[0]) if not row_d.empty else None
        table_text = _format_comparison_table(m_orig, m_asc, m_desc)

        # Render the comparison table in its own dedicated axis
        # (right of the curve plot, never overlaps the curves).
        ax_tbl.text(0.0, 0.95, table_text,
                     transform=ax_tbl.transAxes, ha='left', va='top',
                     fontsize=8.5, family='monospace', color='#222',
                     bbox=dict(boxstyle='round,pad=0.5',
                               facecolor='#fafafa', edgecolor='#bbb',
                               lw=0.6))
        # Small caption above the table
        ax_tbl.text(0.0, 1.0,
                     f'{mdl}: original vs reconstructed',
                     transform=ax_tbl.transAxes,
                     ha='left', va='bottom',
                     fontsize=9.5, fontweight='bold', color='#333')

    # Re-arrange: legend below
    legend_handles = [
        plt.Line2D([0], [0], linestyle='--', color='#444', lw=1.4,
                    label='original (Part-1)'),
        plt.Line2D([0], [0], linestyle=':', color='#4e79a7', lw=1.0,
                    label='envelope · ascending'),
        plt.Line2D([0], [0], linestyle=':', color='#e15759', lw=1.0,
                    label='envelope · descending'),
        plt.Line2D([0], [0], linestyle='-', color='#4e79a7', lw=2.0,
                    label='reconstructed · ascending'),
        plt.Line2D([0], [0], linestyle='-', color='#e15759', lw=2.0,
                    label='reconstructed · descending'),
    ]
    fig.legend(handles=legend_handles,
                loc='lower center', bbox_to_anchor=(0.5, -0.02),
                ncol=5, fontsize=9.5, frameon=False)

    fig.suptitle(f'{country} · {scenario} — smooth reconstruction',
                  fontsize=14, fontweight='bold', y=0.99)
    # Extra right margin so the comparison table fits in the figure
    slug = f'{country.lower().replace(" ", "_")}_{scenario}'
    fig.savefig(CTRY_DIR / f'{slug}_reconstruction.pdf',
                 dpi=160, bbox_inches='tight')
    fig.savefig(CTRY_DIR / f'{slug}_reconstruction.png',
                 dpi=160, bbox_inches='tight')
    plt.close(fig)


print('Generating per-(country, scenario) profiles ...')
groups = failed.groupby(['country', 'scenario']).size().reset_index(name='n')
for _, g in groups.iterrows():
    _plot_country_scenario(g['country'], g['scenario'])
print(f'  wrote {len(groups)} figures to {CTRY_DIR}')

Generating per-(country, scenario) profiles ...


  wrote 23 figures to /Users/xg320/Desktop/IC-Sam-works/paper/Iman-2026/03_feasible_growth/output/figures/country_profiles
